In [ ]:
import os
import certifi
import requests
from dotenv import load_dotenv, find_dotenv

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool


In [2]:
from langchain.agents import create_react_agent, AgentExecutor

In [115]:
# ==============================================================
# LOAD ENV VARIAB from .env file
# ==============================================================
os.environ["SSL_CERT_FILE"] = certifi.where()
#os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
load_dotenv()


OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")


In [8]:
search_tool = TavilySearchResults(max_results=2)

In [131]:
@tool
def get_weather_data(city: str) -> str:
    """
    Get the current weather for a given city.
    """
    url = f"http://api.weatherstack.com/current?access_key={WEATHERSTACK_API_KEY}&query={city}"

    response = requests.get(url)

    data = response.json()
    print(data)
    if "current" in data:

        return (
            f"City: {city}\n"
            f"Temperature: {data['current']['temperature']}°C\n"
            f"Weather Description: {data['current']['weather_descriptions'][0]}\n"
        )
    else:
        return f"Could not retrieve weather data for {city}. Please check the city name and try again." 
    

In [9]:
result = search_tool.invoke("What is the latest news on AI?")
result

[{'url': 'https://www.artificialintelligence-news.com',
  'content': 'Artificial Intelligence\n\nApril 11, 2024\n\n### UK and Canada sign AI compute agreement\n\nArtificial Intelligence\n\nJanuary 31, 2024\n\n#### Machine Learning\n\n### Samsung health AI models analyse wearable biosignal data\n\nHealthcare & Wellness AI\n\nAugust 14, 2026\n\n### Why biological data matters more in AI drug discovery\n\nAI in Action\n\nAugust 3, 2026\n\n### Meta, Microsoft, Nvidia, IBM, and others back open-weight AI\n\nOpen-Source & Democratised AI\n\nJuly 24, 2026\n\n#### Enterprise\n\n### OpenAI president urges enterprises to hasten AI security defences\n\nCybersecurity AI\n\nAugust 18, 2026\n\n### Okta targets AI agent token costs with MCP scoping\n\nData Engineering & MLOps\n\nAugust 13, 2026\n\n### Red Hat, NVIDIA, IBM back project turning AI policy into code\n\nGovernance, Regulation & Policy\n\nAugust 4, 2026 [...] Governance, Regulation & Policy\n\nAugust 4, 2026\n\n#### Industries\n\n### Zucke

In [ ]:
# ==============================================================
# LLM SETUP
# ==============================================================

llm = ChatOpenAI(
    #model="gpt-5.6-luna",
    model="gpt-4.1-nano",
    temperature=1,
    openai_api_key=OPENAI_API_KEY
)

In [40]:
response = llm.invoke("What year it is?")
response

AIMessage(content='The current year is 2024.', response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 12, 'total_tokens': 20, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano', 'system_fingerprint': 'fp_ea5819d2b4', 'finish_reason': 'stop', 'logprobs': None}, id='run-3143be49-9f48-402c-9f46-8e1aa34ab522-0')

In [67]:
# ==============================================================
# PROMPT
# ==============================================================

prompt = hub.pull("hwchase17/react")

c:\Users\raipi\Param\Training\Search-and-weather-AI-Agent-Using-Langchain\langagent\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [68]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [133]:
# ==============================================================
# TOOLS SETUP
# ==============================================================

tools = [search_tool, get_weather_data]


In [134]:
# ==============================================================
# CREATE AGENT
# ==============================================================

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [135]:
# ==============================================================
# EXECUTOR
# ==============================================================

agent_executor = AgentExecutor(
    agent=agent,
      tools=tools,
        verbose=True
    )

In [136]:
# ==============================================================
# RUN
# ==============================================================

response = agent_executor.invoke({
    "input": "What is latest news on USA IRAN war and what is the current weather in Tehran?"
    })




> Entering new AgentExecutor chain...
Action: tavily_search_results_json  
Action Input: latest news on USA IRAN war  [{'url': 'https://www.britannica.com/event/2026-Iran-war', 'content': '2 UAE tankers attacked while transiting Strait of Hormuz, and other news from the Middle East• Aug. 14, 2026, 3:35 PM ET (AP) \n\nWith US-Iran talks stalled, diplomatic efforts expand to unlikely European players• Aug. 14, 2026, 2:37 PM ET (AP) \n\nRubio and Bessent stayed on decoy flight from Turkey as Trump switched planes, official says• Aug. 12, 2026, 9:44 PM ET (AP) \n\nThe Latest: White House press secretary Karoline Leavitt will leave at the month’s end• Aug. 12, 2026, 7:56 PM ET (AP) \n\n Show less'}, {'url': 'https://www.bbc.com/news/topics/cx2jyv8j8gwt', 'content': '# News\n\n# Iran war\n\nFILE PHOTO: The Epaminondas container ship, in Naples, Italy March 12, 2025, in this picture obtained by Reuters on April 22, 2026.\n\n## Iran says reopening Strait of Hormuz \'impossible\' if US blocka

In [137]:
print(response["output"])

The latest news indicates ongoing tensions between the USA and Iran, including stalled talks, military movements, and Iran's claims about the Strait of Hormuz. Regarding the weather in Tehran, it is currently clear with a temperature of approximately 33°C.
